In [3]:
import numpy as np
import matplotlib.pyplot as plt

# 首先尝试导入tensorflow，如果失败则给出错误信息
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
except Exception as e:
    print("Error importing TensorFlow:", e)
    raise

# 加载数据
print("Loading data...")
data = np.load('E:\\My struggle\\SVM 手写数字\\mnist.npz')
x_train, y_train = data['x_train'], data['y_train']
x_test, y_test = data['x_test'], data['y_test']

# 数据预处理
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# One-hot编码
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 使用Sequential API构建模型（更简单的方式）
model = tf.keras.Sequential([
    # 第一个卷积块
    tf.keras.layers.Conv2D(32, (7, 7), padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # 第二个卷积块
    tf.keras.layers.Conv2D(64, (5, 5), padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # 展平层
    tf.keras.layers.Flatten(),
    
    # 全连接层
    tf.keras.layers.Dense(1024),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.1),
    
    # 输出层
    tf.keras.layers.Dense(10, activation='softmax')
])

# 打印模型结构
model.summary()

# 编译模型
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 定义回调函数
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

# 训练模型
print("\nTraining model...")
history = model.fit(
    x_train, y_train,
    batch_size=100,
    epochs=20,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# 评估模型
print("\nEvaluating model...")
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_accuracy*100:.2f}%")

# 保存模型
model.save('mnist_cnn_model.h5')
print("Model saved as 'mnist_cnn_model.h5'")

# 可视化训练过程
plt.figure(figsize=(12, 4))

# 准确率曲线
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# 损失曲线
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 测试预测
def predict_and_display(images, num_samples=5):
    plt.figure(figsize=(15, 3))
    for i in range(num_samples):
        # 预测
        pred = model.predict(images[i:i+1])
        pred_digit = np.argmax(pred[0])
        confidence = pred[0][pred_digit]
        
        # 显示图像
        plt.subplot(1, 5, i+1)
        plt.imshow(images[i].reshape(28, 28), cmap='gray')
        plt.title(f'Pred: {pred_digit}\n{confidence*100:.1f}%')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# 显示一些预测结果
print("\nDisplaying some predictions...")
predict_and_display(x_test)

Error importing TensorFlow: Symbol arg_max is already exposed as ().


SymbolAlreadyExposedError: Symbol arg_max is already exposed as ().